# TP 1 — La naturaleza de la econometría y los datos económicos

**Alumna:** Lucía Vázquez
**Materia:** Econometría I
**Fecha:** 31/08/2026
**Referencia:** Wooldridge, *Introducción a la econometría*, capítulo 1
**Entregable:** Ejercicios 1.2, 1.5, 1.6, 1.7 y 1.8

**Nota sobre el formato.** Este cuaderno sigue la plantilla de entrega
(Anexo II): Encabezado, 1. Pregunta económica, 2. Datos, 3. Especificación,
4. Estimación, 5. Inferencia, 6. Interpretación económica, 7. Limitaciones.
La plantilla está pensada para una única estimación por entrega; como este
TP pide cinco ejercicios independientes —varios de ellos sin datos ni
modelo estimado, porque el capítulo 1 todavía no cubre regresión ni
contraste de hipótesis— la estructura se aplica *dentro de cada ejercicio*,
y se marca "No aplica" (con la razón) en las secciones que no correspondan
a un ejercicio puntual, en lugar de forzar un modelo o una prueba que el
enunciado no pide.

Cada ejercicio con cálculos muestra primero el código en R (bloque de
referencia, no ejecutado en este notebook) y luego en Python (celda
ejecutada, con la salida real debajo). Los datos provienen del paquete
`wooldridge`, disponible en R (`library(wooldridge)`) y en Python
(`import wooldridge as woo`).

In [1]:
# Configuración (Python)
import numpy as np
import pandas as pd
import wooldridge as woo
import statsmodels.formula.api as smf
from scipy import stats

pd.set_option('display.precision', 4)


---
# Ejercicio 1.2 (Wooldridge C1.2, `bwght`)

*Consigna: informar cuántas mujeres hay en la muestra, cuántas fumaron
durante el embarazo y el promedio de cigarrillos diarios (`cigs`); repetir
el promedio solo entre las fumadoras y explicar la diferencia.*

### 1. Pregunta económica
¿Cuánto fuma, en promedio, una mujer durante el embarazo, y cambia esa
respuesta según se calcule sobre toda la población de nacimientos o solo
sobre quienes efectivamente fuman? Importa porque el consumo de cigarrillos
durante el embarazo es un determinante conocido del peso al nacer, y una
medida mal elegida puede subestimar la exposición real de los bebés
afectados.

### 2. Datos
`bwght` (paquete `wooldridge`). Corte transversal de nacimientos en Estados
Unidos, 1,388 observaciones. Unidad de observación: nacimiento (madre que
dio a luz). Variable utilizada: `cigs`, cigarrillos fumados por día durante
el embarazo (cigarrillos/día).

### 3. Especificación
No hay una relación entre una variable dependiente y una independiente con
un término de error: se trata de dos parámetros poblacionales puramente
descriptivos,
$$\mu = E(\text{cigs}) \quad \text{y} \quad \mu_s = E(\text{cigs} \mid \text{cigs} > 0),$$
estimados por sus análogos muestrales, la media sobre toda la muestra y la
media condicional entre fumadoras.

**Código en R**

```r
library(wooldridge)
data(bwght, package = "wooldridge")

nrow(bwght)                          # tamaño de la muestra
sum(bwght$cigs > 0)                  # cuántas fumaron
mean(bwght$cigs)                     # estimador de mu
mean(bwght$cigs[bwght$cigs > 0])     # estimador de mu_s
```

In [2]:
# 4. Estimación — Código en Python
bwght = woo.dataWoo('bwght')

n = len(bwght)
fumadoras = (bwght['cigs'] > 0).sum()
mu_hat = bwght['cigs'].mean()
mu_s_hat = bwght.loc[bwght['cigs'] > 0, 'cigs'].mean()

print('n (muestra completa):', n)
print('Mujeres que fumaron durante el embarazo:', fumadoras)
print('mu_hat  = media de cigs, toda la muestra  =', round(mu_hat, 3))
print('mu_s_hat = media de cigs, solo fumadoras   =', round(mu_s_hat, 3))


n (muestra completa): 1388
Mujeres que fumaron durante el embarazo: 212
mu_hat  = media de cigs, toda la muestra  = 2.087
mu_s_hat = media de cigs, solo fumadoras   = 13.665


**Resultado, formato ecuación:**

$$\hat\mu = 2.087 \ \text{cig/día} \quad (n = 1{,}388) \qquad\qquad \hat\mu_s = 13.665 \ \text{cig/día} \ (n_s = 212)$$

### 5. Inferencia
No aplica. El ejercicio no define dos poblaciones cuya diferencia deba
contrastarse ni una hipótesis de referencia: pide elegir cuál de dos
promedios responde mejor a la pregunta planteada, no una prueba estadística
sobre una diferencia.

### 6. Interpretación económica
La muestra tiene 1,388 mujeres que dieron a luz; 212 (15.3%) reportan haber
fumado durante el embarazo. El promedio de cigarrillos diarios sobre toda la
muestra, 2.09, mezcla el 84.7% que fumó cero cigarrillos con quienes fuman
mucho, y por eso no describe la intensidad de consumo de nadie en
particular. El promedio condicional en `cigs > 0`, 13.67 cigarrillos
diarios, sí aísla al grupo relevante y es la cifra que responde "cuánto
fuma una mujer que fuma": una intensidad de consumo considerable, más de
media cajetilla diaria.

### 7. Limitaciones
Ambas cifras dependen de un dato autorreportado sobre una conducta
estigmatizada socialmente durante el embarazo, por lo que es esperable una
subdeclaración tanto de quién fuma (212 podría ser un piso) como de cuánto
fuma cada una (13.67 también podría ser un piso). La variable que falta es
una medida objetiva de exposición (por ejemplo, cotinina en sangre u orina);
en su ausencia, el sesgo de subdeclaración empuja ambos promedios hacia
abajo respecto del consumo verdadero.

---
# Ejercicio 1.5 (`wage1`)

*Consigna: calcular el salario promedio por hora de varones y mujeres,
nombrar tres variables del dataset que podrían explicar parte de la brecha
sin invocar discriminación e indicar en qué dirección operaría cada una.*

### 1. Pregunta económica
¿Cuánto difiere el salario horario promedio entre varones y mujeres en la
muestra, y es esa diferencia demasiado grande para ser producto del azar
muestral? Es el punto de partida habitual para estudiar la brecha salarial
de género, aunque por sí sola la comparación bruta de medias no permite
atribuir la diferencia a discriminación.

### 2. Datos
`wage1` (paquete `wooldridge`). Corte transversal, Current Population Survey
de Estados Unidos, 1976, 526 individuos empleados. Unidad de observación:
individuo. Variables utilizadas: `wage` (salario, USD/hora de 1976),
`female` (1 = mujer, 0 = varón), y como controles de referencia `educ`
(años de educación), `exper` (años de experiencia potencial) y `tenure`
(años de antigüedad en el empleo actual).

### 3. Especificación
La comparación de medias entre dos grupos es algebraicamente una regresión
simple con un regresor binario:
$$\text{wage}_i = \beta_0 + \beta_1\, \text{female}_i + u_i,$$
donde $\beta_0 = E(\text{wage} \mid \text{female}=0)$ es el salario promedio
de los varones, $\beta_0+\beta_1 = E(\text{wage}\mid \text{female}=1)$ el de
las mujeres, y $\beta_1$ la diferencia promedio bruta entre ambos grupos.
El supuesto $E(u_i \mid \text{female}_i) = 0$ es precisamente lo que está en
duda: si `female` está correlacionada con otros determinantes del salario
omitidos de esta especificación (antigüedad, ocupación), el supuesto falla y
$\hat\beta_1$ no puede leerse como el efecto "puro" de ser mujer.

**Código en R**

```r
library(wooldridge)
data(wage1, package = "wooldridge")

# Comparación de medias
aggregate(wage ~ female, data = wage1, FUN = mean)

# Especificación equivalente como regresión simple
m <- lm(wage ~ female, data = wage1)
summary(m)
```

In [3]:
# 4. Estimación — Código en Python
wage1 = woo.dataWoo('wage1')

medias = wage1.groupby('female')['wage'].mean()
brecha_abs = medias[0] - medias[1]
brecha_pct = 100 * brecha_abs / medias[0]
print(medias)
print(f'\nBrecha: {brecha_abs:.3f} USD/hora ({brecha_pct:.1f}% del salario de los varones)')

m = smf.ols('wage ~ female', data=wage1).fit()
print('\nRegresión wage ~ female:')
print(m.params.round(4))
print(m.bse.round(4))
print('n =', int(m.nobs), ' R2 =', round(m.rsquared, 3))


female
0    7.0995
1    4.5877
Name: wage, dtype: float64

Brecha: 2.512 USD/hora (35.4% del salario de los varones)

Regresión wage ~ female:
Intercept    7.0995
female      -2.5118
dtype: float64
Intercept    0.2100
female       0.3034
dtype: float64
n = 526  R2 = 0.116


**Ecuación estimada, formato del libro:**

$$\widehat{\text{wage}} = \underset{(0.2100)}{7.0995} \ \underset{(0.3034)}{-\, 2.5118 \ \text{female}}$$
$$n = 526, \quad R^2 = 0.116$$

### 5. Inferencia
$H_0: \beta_1 = 0$ (no hay diferencia salarial promedio entre varones y
mujeres) vs. $H_a: \beta_1 \neq 0$. Estadístico $t = -2.5118/0.3034 =
-8.28$ con $524$ grados de libertad; valor $p \approx 1.0\times10^{-15}$.
Se rechaza $H_0$ ampliamente al 1%: la brecha observada es demasiado grande
y demasiado precisa para atribuirla al azar muestral.

### 6. Interpretación económica
El salario promedio es 7.10 USD/hora para los varones y 4.59 USD/hora para
las mujeres: una brecha de 2.51 USD/hora, un 35.4% del salario masculino.
Es una magnitud económicamente grande y, según la inferencia anterior,
estadísticamente distinguible de cero. Sin invocar discriminación, tres
variables de `wage1` explican parte de ella: **`tenure`** (los varones
promedian 6.47 años de antigüedad contra 3.62 de las mujeres, y el salario
suele crecer con la antigüedad); la **distribución ocupacional** (44.9% de
los varones está en ocupaciones profesionales contra 27.8% de las mujeres,
mientras ellas están sobrerrepresentadas en clerical, 30.6% vs. 4.0%, y
servicios, 19.8% vs. 8.8%); y **`exper`** (17.56 años promedio para varones
contra 16.43 para mujeres, en la misma dirección aunque de menor magnitud).

### 7. Limitaciones
Esta comparación bruta no puede leerse como el efecto causal de ser mujer
sobre el salario. La variable omitida más concreta es `tenure`: al
incorporarla,
$$\widehat{\text{wage}} = 6.136 \underset{(0.295)}{-\,2.087\,\text{female}} + \underset{(0.020)}{0.149\,\text{tenure}}, \quad n=526,$$
el coeficiente de `female` pasa de $-2.512$ a $-2.087$. Como las mujeres de
la muestra tienen sistemáticamente menos antigüedad y la antigüedad se
asocia positivamente con el salario, omitir `tenure` **sesga el coeficiente
de `female` hacia abajo** (lo hace más negativo de lo que sería
manteniendo la antigüedad constante): parte de los 2.51 dólares de brecha
bruta refleja la menor antigüedad promedio de las mujeres en la muestra, no
un trato salarial distinto dentro del mismo puesto y la misma antigüedad.

---
# Ejercicio 1.6 (aproximación logarítmica)

*Consigna: comparar el cambio porcentual exacto y la aproximación
logarítmica para 6,000 vs. 5,500 dólares de gasto por alumno, repetir con
5,500 vs. 11,000, y formular una regla práctica de cuándo usar la
aproximación.*

### 1. Pregunta económica
¿Qué tan confiable es la aproximación $\%\Delta x \approx 100\,[\log(x_1) -
\log(x_0)]$ para medir cambios porcentuales en el gasto educativo por
alumno, y en qué dirección se equivoca cuando el cambio es grande? Importa
porque esta aproximación es la base de leer los coeficientes de variables
en logaritmos directamente como cambios porcentuales, algo que se hace
constantemente en econometría aplicada.

### 2. Datos
No aplica: se trabaja con dos pares de valores hipotéticos de gasto por
alumno (6,000 vs. 5,500 y 5,500 vs. 11,000 dólares), no con una muestra.

### 3. Especificación
Cambio porcentual exacto y su aproximación de primer orden, con
$g = (x_1-x_0)/x_0$ el cambio proporcional:
$$\%\Delta x_{\text{exacto}} = 100\,\frac{x_1-x_0}{x_0} = 100\,g, \qquad
\%\Delta x_{\text{log}} = 100\,[\log(x_1)-\log(x_0)] = 100\,\log(1+g) \approx 100\,g \ \text{para } g \text{ pequeño}.$$

**Código en R**

```r
cambio <- function(x0, x1) {
  exacto <- 100 * (x1 - x0) / x0
  aproximado <- 100 * (log(x1) - log(x0))
  c(exacto = exacto, aproximado = aproximado, error = aproximado - exacto)
}

cambio(5500, 6000)    # de 5,500 a 6,000
cambio(5500, 11000)   # de 5,500 a 11,000
```

In [4]:
# 4. Estimación — Código en Python
def cambio(x0, x1):
    exacto = 100 * (x1 - x0) / x0
    aproximado = 100 * (np.log(x1) - np.log(x0))
    return exacto, aproximado, aproximado - exacto

for x0, x1, etiqueta in [(5500, 6000, '5,500 a 6,000'),
                          (5500, 11000, '5,500 a 11,000')]:
    exacto, aproximado, error = cambio(x0, x1)
    print(f'{etiqueta}: exacto={exacto:.2f}%, log={aproximado:.2f}%, error={error:.2f} p.p.')


5,500 a 6,000: exacto=9.09%, log=8.70%, error=-0.39 p.p.
5,500 a 11,000: exacto=100.00%, log=69.31%, error=-30.69 p.p.


### 5. Inferencia
No aplica: no hay una muestra ni un parámetro poblacional que estimar a
partir de datos con ruido; ambas cantidades son cálculos determinísticos
sobre valores dados.

### 6. Interpretación económica
Para el cambio de 5,500 a 6,000 dólares, el cambio exacto es 9.09% y la
aproximación logarítmica da 8.70%: un error de menos de medio punto
porcentual. Para el cambio de 5,500 a 11,000, el cambio exacto es 100% y
la aproximación da apenas 69.31%: un error de casi 31 puntos porcentuales.
La aproximación se degrada a medida que el cambio relativo se aleja de
cero porque $\log(1+g)\approx g$ es una aproximación de Taylor de primer
orden válida solo para $g$ pequeño; cuando $x$ se duplica, $g=1$ ya no es
"pequeño" y la aproximación lineal deja de servir. **Regla práctica:**
usarla con confianza para cambios de hasta 10-15% (error por debajo de 1
punto porcentual); para cambios mayores al 20-30% reportar el cambio
porcentual exacto.

### 7. Limitaciones
Es una limitación matemática, no de datos: como $\log$ es cóncavo,
$\log(1+g) \le g$ para todo $g$, con igualdad solo en $g=0$. Esto implica
una dirección de error sistemática: para aumentos ($g>0$) la aproximación
**subestima** el crecimiento verdadero, y para caídas ($g<0$) la
aproximación logarítmica es más negativa que el cambio exacto, es decir
**sobrestima la magnitud** de la caída. Cualquier lectura de un
coeficiente en logaritmos como "cambio porcentual" hereda este mismo
sesgo cuando el coeficiente implica cambios grandes.

---
# Ejercicio 1.7 (estructura de datos)

*Consigna: clasificar cuatro conjuntos de datos según su estructura (corte
transversal, serie de tiempo, corte transversal combinado o panel).*

### 1. Pregunta económica
¿Qué estructura de datos corresponde a cada uno de los cuatro conjuntos
descritos, y por qué importa distinguirlas? Importa porque el método
econométrico válido —por ejemplo, corregir por autocorrelación serial en
series de tiempo, o por heterogeneidad no observada en un panel— depende
críticamente de esa clasificación.

### 2. Datos
No aplica: el ejercicio describe hipotéticamente cuatro conjuntos de datos;
alcanza con identificar la unidad de observación y la dimensión temporal
declaradas en cada enunciado, sin trabajar sobre una base concreta.

### 3. Especificación
No aplica: no hay una ecuación poblacional que especificar, sino una
clasificación taxonómica basada en dos preguntas — ¿hay más de un momento
del tiempo? y, si los hay, ¿son las mismas unidades en cada uno?

### 4. Estimación
No aplica.

### 5. Inferencia
No aplica.

### 6. Interpretación económica

**a) Precio de cierre diario del dólar mayorista, 2015–2025.**
Serie de tiempo. La unidad de observación es una sola unidad —el tipo de
cambio mayorista— seguida repetidamente en el tiempo (un valor por día
durante diez años); no hay una sección cruzada de unidades distintas en
ningún momento.

**b) Ingreso, edad y educación de 3,000 hogares relevados por la EPH del
segundo trimestre de 2024.**
Corte transversal. La unidad de observación es el hogar, y los 3,000 se
observan en un único momento (2T-2024); no hay dimensión temporal.

**c) PBI, población y esperanza de vida de 180 países en 1990, 2000, 2010 y
2020.**
Panel. La unidad de observación —el país— se sigue explícitamente en varios
momentos del tiempo, y en cada uno se trata de las mismas 180 unidades, no
de muestras nuevas.

**d) Ventas y empleo de 500 empresas industriales, relevadas en 2019 y
2023, sin garantía de que sean las mismas empresas.**
Corte transversal combinado (*pooled cross section*). Hay dos momentos del
tiempo, pero, a diferencia de (c), sin seguimiento garantizado de las
mismas unidades: son dos muestras independientes de empresas. Esa frase
final del enunciado es exactamente la pista que descarta panel.

### 7. Limitaciones
La clasificación por sí sola no garantiza que un método de estimación sea
válido: un panel como (c) puede estar desbalanceado (no todos los países
reportan los cuatro años) y requiere además supuestos sobre la
heterogeneidad no observada entre países; una serie de tiempo como (a)
puede no ser estacionaria; y el corte transversal combinado de (d) solo es
informativo sobre cambios agregados en el tiempo, nunca sobre cambios
dentro de una misma empresa, precisamente porque no se garantiza que sean
las mismas unidades.

---
# Ejercicio 1.8 (Wooldridge, problema 1.2 — capacitación y productividad)

*Consigna: redactar el experimento subyacente al efecto de la capacitación
laboral sobre la productividad, evaluar la independencia entre la decisión
de capacitar y las características de los trabajadores, nombrar un factor
ajeno a los trabajadores que afecte la productividad, y discutir si una
correlación positiva entre producción y capacitación establece
causalidad.*

### 1. Pregunta económica
¿La capacitación laboral aumenta la productividad de los trabajadores, y
puede esa relación estimarse a partir de la decisión observacional de una
empresa de capacitar (o no) a su personal?

### 2. Datos
No aplica: es un problema conceptual (Wooldridge, problema 1.2), no un
ejercicio de cómputo; no requiere ninguna base de datos.

### 3. Especificación — el experimento subyacente (parte a)
El experimento ideal consiste en observar a los mismos trabajadores (o
empresas) en dos estados del mundo idénticos en todo excepto en si reciben
capacitación: mismo capital humano, mismo acceso a tecnología y maquinaria,
misma calidad de gestión, mismas condiciones de mercado. Se asignaría
aleatoriamente a una parte de ellos al programa de capacitación y a la otra
no, y se compararía luego su productividad (por ejemplo, producto por hora
trabajada) manteniendo todo lo demás constante. La aleatorización es lo que
garantiza que "todo lo demás" quede balanceado entre ambos grupos, en
promedio, sin necesidad de observarlo ni medirlo.

### 4. Estimación
No aplica: sin datos no hay estimador que calcular. Lo que sí puede
señalarse es qué impide construir ese estimador a partir de datos
observacionales (parte b): la decisión de una empresa de capacitar rara vez
es independiente de las características de sus trabajadores. Dos
características **medibles** que romperían esa independencia: el nivel
educativo previo del personal y su antigüedad en el puesto. Dos **no
medibles**: la motivación o habilidad innata de los trabajadores y la
calidad de la gestión gerencial de la empresa.

### 5. Inferencia
No aplica: no hay una muestra sobre la cual contrastar una hipótesis.

### 6. Interpretación económica
**c) Un factor ajeno a los trabajadores que afecte la productividad:** la
calidad del capital físico y la tecnología con la que trabajan (maquinaria,
software, organización del proceso productivo). Dos plantas con
trabajadores idénticos pueden diferir mucho en productividad simplemente
porque una tiene equipamiento más moderno.

### 7. Limitaciones — ¿la correlación establece causalidad? (parte d)
No. Es el problema central del capítulo. La correlación positiva entre
producción y capacitación es compatible con al menos dos historias no
causales: **causalidad inversa**, donde las empresas más rentables o
productivas por otras razones son las que tienen recursos ociosos para
capacitar a su personal (la productividad previa genera la capacitación, no
al revés); y una **variable omitida**, la calidad gerencial, que afecta
simultáneamente la probabilidad de capacitar y el nivel de productividad,
generando la correlación aunque la capacitación en sí no tuviera ningún
efecto causal. Para atribuir la diferencia a la capacitación haría falta
algo que se aproxime a una asignación aleatoria del tratamiento, o al menos
controlar explícitamente por gestión, tecnología y características previas
de los trabajadores.